# 第 8 章：工程部署及性能分析 — 章节介绍

## 1. 前置要求

学习本章节之前，请确保你已具备以下能力或已完成相关学习：

- **前置课程**：已完成本课程 `01`-`07` 全部章节（数据组织与内存访问、并行计算、矩阵乘法分块、栈的表达式求值、算子工程与性能优化基础等）。
- **Ascend C 基础**：了解自定义算子工程结构（op_host / op_kernel）、Tiling 的作用、Kernel 直调或 aclnn 调用方式。
- **环境**：具备可用的 CANN 9.0.0 + Atlas A2（Ascend 910B3）环境，或 CANNLab 云开发环境（镜像模板 `cann_9.0.0_py3.11-A2-arm`）。

> 若对前序章节内容不熟悉，建议先完成 `01`-`07` 全部课程再继续。

## 2. 章节目标

本实验以**注意力算子**（QKᵀ → scale → softmax → AV）为载体，打通 Ascend C 自定义算子的工程全流程：

1. 使用 `msOpGen` 基于算子原型定义（ops.json）生成算子工程；
2. 完成 Host 侧（Tiling / InferShape / InferDataType）与 Kernel 侧实现；
3. 编译 → 打包 → 部署到 CANN 用户目录，并通过 aclnn 接口单算子调用、精度验证；
4. 使用 `msProf` 上板采集性能数据并解读报告（Task Duration / 流水利用率）；
5. 推导并实测注意力算子的 O(S²) 复杂度，对比 Flash Attention 理论；
6. 认识 `msSanitizer` 异常检测与 `msDebug` 调试工具。

## 3. 学习路径与内容导航

<table style="text-align: left; margin-left: 0;">
  <thead>
    <tr>
      <th>Notebook</th>
      <th>内容</th>
      <th>预计耗时</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><a href="08.01_chapter_intro.ipynb">08.01 章节介绍</a></td>
      <td>前置要求 / 目标 / 导航</td>
      <td>5 分钟</td>
    </tr>
    <tr>
      <td><a href="08.02_attention_operator_lab.ipynb">08.02 动手实验</a></td>
      <td>全流程 6 步骤实操</td>
      <td>60-90 分钟</td>
    </tr>
    <tr>
      <td><a href="08.03_chapter_test.ipynb">08.03 章节实践</a></td>
      <td>综合编程实践 + 知识测验</td>
      <td>30-60 分钟</td>
    </tr>
  </tbody>
</table>

## 4. 注意力算子计算链

```text
scores[i][j] = Σₖ q[i][k] · kt[k][j] · scale        # QKᵀ + scale，scale = 1/√D
P[i][j]     = softmax(scores[i][:])                 # 行 softmax（max 减 / exp / sum 归一）
o[i][j]     = Σₖ P[i][k] · v[k][j]                  # AV
```

总计算量：**FLOPs ≈ 4·S²·D**（QKᵀ 与 AV 各 2·S²·D），即 **O(S²)** 复杂度 —— 这是大模型推理的经典性能瓶颈，也是本章复杂度演示的核心。

## 5. 工程全流程

<div style="text-align: left;">
  <img src="./images/engineering_flow.svg" alt="工程全流程示意图" width="720">
</div>

## 6. 注意力数据流与多核切分

<div style="text-align: left;">
  <img src="./images/attention_dataflow.svg" alt="注意力算子数据流" width="720">
</div>

**下一步**：打开 [08.02_attention_operator_lab.ipynb](08.02_attention_operator_lab.ipynb) 开始动手实验。